# Generating ONNX Training graph

Generating artifacts based on forward-only graph and the specified loss function using onnxblock library

In [12]:
from pathlib import Path

import numpy as np
import onnx
import onnxruntime.training.onnxblock as onnxblock

In [13]:
# Creating a class with a Loss function
class CustomTrainingBlock(onnxblock.TrainingBlock):
    def __init__(self):
        super(CustomTrainingBlock, self).__init__()
        self.loss = onnxblock.loss.CrossEntropyLoss()

    def build(self, output_name):
        return self.loss(output_name), output_name

In [14]:
model_name = "fastvit_sa12"
runtime = "hailo8_optim_3_comp_0"  # cpu, hailo8_optim_0_comp_0
classes_num = 100
model_frozen_layer_num = {
    'resnet18': 12,
    'resnet34': 20,
    'resnet50': 20,
    'mobilenetv2_100': 21,
    'efficientnet_b2': 27,
    'visformer_tiny': 23,
    'lcnet_100': 19,
    'semnasnet_100': 20,
    'mobilenetv3_large_100': 22,
    'resnet10t': 14,
    'convnext_tiny': 32,
    'vit_tiny_patch16_224': 18,
    'tf_efficientnet_lite4': 34,
    'fastvit_sa12': 9,
}
frozen_layer_num = model_frozen_layer_num[model_name]

artifacts_path = Path(f"../artifacts/{model_name}/frozen_layer_{frozen_layer_num}_classes_{classes_num}/{runtime}")
artifacts_path.mkdir(parents=True, exist_ok=True)
onnx_artifacts_path = artifacts_path.parent / f"{model_name}.onnx" if runtime == "cpu" else artifacts_path / f"{model_name}_quantized_model_hailo.onnx"

onnx_model = onnx.load_model(onnx_artifacts_path)

# for param in onnx_model.graph.initializer:
#     print(param.name)

In [15]:
# Patch MaxPool nodes to include indices output for ORT training gradients
from onnx import TensorProto, helper

existing_vi = {vi.name for vi in onnx_model.graph.value_info}
existing_vi.update(vi.name for vi in onnx_model.graph.output)
added = 0

for node in onnx_model.graph.node:
    if node.op_type != "MaxPool":
        continue
    if len(node.output) != 1:
        continue
    indices_name = f"{node.output[0]}_indices"
    node.output.append(indices_name)
    if indices_name in existing_vi:
        continue
    out_vi = None
    for vi in list(onnx_model.graph.value_info) + list(onnx_model.graph.output):
        if vi.name == node.output[0]:
            out_vi = vi
            break
    if out_vi is not None and out_vi.type.tensor_type.HasField("shape"):
        dims = [
            d.dim_value if d.dim_value > 0 else d.dim_param
            for d in out_vi.type.tensor_type.shape.dim
        ]
        vi = helper.make_tensor_value_info(indices_name, TensorProto.INT64, dims)
    else:
        vi = helper.make_tensor_value_info(indices_name, TensorProto.INT64, None)
    onnx_model.graph.value_info.append(vi)
    existing_vi.add(indices_name)
    added += 1

print("MaxPool indices outputs added:", added)

MaxPool indices outputs added: 0


In [16]:
# # Patch LayerNormalization/GroupNormalization to include mean and invstd outputs
# from onnx import TensorProto, helper

# existing_vi = {vi.name for vi in onnx_model.graph.value_info}
# existing_vi.update(vi.name for vi in onnx_model.graph.output)
# added = 0

# def _add_ln_outputs(node, out0_name):
#     global added
#     mean_name = f"{out0_name}_mean"
#     invstd_name = f"{out0_name}_invstd"
#     node.output.extend([mean_name, invstd_name])
#     for extra_name in (mean_name, invstd_name):
#         if extra_name in existing_vi:
#             continue
#         vi = helper.make_tensor_value_info(extra_name, TensorProto.FLOAT, None)
#         onnx_model.graph.value_info.append(vi)
#         existing_vi.add(extra_name)
#     added += 1

# for node in onnx_model.graph.node:
#     if node.op_type not in {"LayerNormalization", "GroupNormalization"}:
#         continue
#     if len(node.output) != 1:
#         continue
#     _add_ln_outputs(node, node.output[0])

# print("Layer/GroupNorm extra outputs added:", added)

In [17]:
def convert_bn_to_static_scale_shift(model):
    """
    Replaces BatchNormalization nodes (and attempts to fold other normalization ops
    if they carry running mean/var as inputs) with explicit Mul and Add nodes
    using their pre-trained running mean, variance, gamma, and beta.
    """
    initializer_dict = {tensor.name: tensor for tensor in model.graph.initializer}
    new_nodes = []
    new_initializers = []

    for idx, node in enumerate(model.graph.node):
        op = node.op_type
        foldable = op == "BatchNormalization" or (
            op in {"InstanceNormalization", "LayerNormalization", "GroupNormalization"}
            and len(node.input) >= 5
        )

        if not foldable:
            if op in {"InstanceNormalization", "LayerNormalization", "GroupNormalization"}:
                print(
                    f"Skipping conversion for {node.name} ({node.op_type}): "
                    "no running mean/var to fold."
                )
            new_nodes.append(node)
            continue

        try:
            # BN-like inputs: X, scale (gamma), B (beta), input_mean, input_var
            x_name = node.input[0]
            gamma_name = node.input[1]
            beta_name = node.input[2]
            mean_name = node.input[3]
            var_name = node.input[4]
            y_name = node.output[0]

            # Extract epsilon attribute (defaults to 1e-5 if not present)
            epsilon = 1e-5
            for attr in node.attribute:
                if attr.name == "epsilon":
                    epsilon = attr.f

            # Retrieve numpy arrays from initializers
            gamma = onnx.numpy_helper.to_array(initializer_dict[gamma_name])
            beta = onnx.numpy_helper.to_array(initializer_dict[beta_name])
            mean = onnx.numpy_helper.to_array(initializer_dict[mean_name])
            var = onnx.numpy_helper.to_array(initializer_dict[var_name])

            # Calculate static Scale (A) and Shift (B)
            # Math: Output = (X - mean) / sqrt(var + eps) * gamma + beta
            #             = X * (gamma / sqrt(var + eps)) + (beta - mean * gamma / sqrt(var + eps))
            scale_val = gamma / np.sqrt(var + epsilon)
            shift_val = beta - (mean * scale_val)

            # Determine a sensible reshape for broadcasting:
            # - If 1D per-channel parameters and model uses NCHW conv outputs, reshape to (1, C, 1, 1)
            # - Otherwise keep as-is (e.g., LayerNorm over last dim)
            def _reshape_for_broadcast(arr):
                if arr.ndim == 1:
                    return arr.reshape(1, -1, 1, 1).astype(np.float32)
                return arr.astype(np.float32)

            scale_val = _reshape_for_broadcast(scale_val)
            shift_val = _reshape_for_broadcast(shift_val)

            # Create unique names for the new static weights
            node_tag = node.name if node.name else f"{op}_{idx}"
            static_scale_name = f"{node_tag}_static_scale"
            static_shift_name = f"{node_tag}_static_shift"
            intermediate_tensor_name = f"{node_tag}_mul_output"

            # Create ONNX initializers for Scale and Shift
            new_initializers.append(
                onnx.numpy_helper.from_array(scale_val, static_scale_name)
            )
            new_initializers.append(
                onnx.numpy_helper.from_array(shift_val, static_shift_name)
            )

            # Create Mul and Add nodes to replace the normalization node
            mul_node = onnx.helper.make_node(
                "Mul",
                inputs=[x_name, static_scale_name],
                outputs=[intermediate_tensor_name],
                name=f"{node_tag}_static_mul",
            )
            add_node = onnx.helper.make_node(
                "Add",
                inputs=[intermediate_tensor_name, static_shift_name],
                outputs=[y_name],
                name=f"{node_tag}_static_add",
            )

            # Insert in place to preserve topological ordering
            new_nodes.extend([mul_node, add_node])
        except KeyError:
            # Missing expected initializers; skip folding for this node
            print(
                f"Skipping conversion for {node.name} ({op}): "
                "required initializers not found."
            )
            new_nodes.append(node)

    # RepeatedCompositeContainer in some onnx/protobuf builds lacks clear()
    del model.graph.node[:]
    model.graph.node.extend(new_nodes)
    model.graph.initializer.extend(new_initializers)

    return model

In [18]:
# Convert BN-like nodes to static math operations
onnx_model = convert_bn_to_static_scale_shift(onnx_model)

def _reachable_initializers(model, output_names):
    init_names = {init.name for init in model.graph.initializer}
    produced_by = {}
    for node in model.graph.node:
        for out_name in node.output:
            produced_by[out_name] = node
    visited = set()
    stack = list(output_names)
    while stack:
        name = stack.pop()
        if name in visited:
            continue
        visited.add(name)
        node = produced_by.get(name)
        if not node:
            continue
        for in_name in node.input:
            if in_name and in_name not in visited:
                stack.append(in_name)
    return init_names.intersection(visited)

def _is_trainable_param(name):
    return name.endswith(".weight") or name.endswith(".bias")

training_block = CustomTrainingBlock()
output_names = [output.name for output in onnx_model.graph.output]
if not output_names:
    raise ValueError("ONNX model has no graph outputs to attach loss to.")
if len(output_names) > 1:
    print(f"Warning: multiple graph outputs found; using the first: {output_names[0]}")
output_name = output_names[0]
reachable_inits = _reachable_initializers(onnx_model, [output_name])
if not reachable_inits:
    print("Warning: no reachable initializers found for the selected output.")

for param in onnx_model.graph.initializer:
    if param.name not in reachable_inits:
        print(param.name.ljust(40), "\t--->\tunused; freezing")
        training_block.requires_grad(param.name, False)
        continue
    if "frozen_features" in param.name or "bn" in param.name or "downsample.1" in param.name or "backbone" in param.name:
        print(param.name.ljust(40), "\t--->\tfrozen")
        training_block.requires_grad(param.name, False)
    elif _is_trainable_param(param.name):
        print(param.name.ljust(40), "\t--->\tlearnable")
        training_block.requires_grad(param.name, True)
    else:
        print(param.name.ljust(40), "\t--->\tconst; freezing")
        training_block.requires_grad(param.name, False)

# Building training graph and eval graph
model_params = None
with onnxblock.base(onnx_model):
    build_output = training_block(output_name)
    print("Graph output nodes:", build_output)
    training_model, eval_model = training_block.to_model_proto()
    model_params = training_block.parameters()

# Building the optimizer graph
optimizer_block = onnxblock.optim.AdamW()
with onnxblock.empty_base() as accessor:
    _ = optimizer_block(model_params)
    optimizer_model = optimizer_block.to_model_proto()

post_output.weight                       	--->	learnable
post_output.bias                         	--->	learnable
Graph output nodes: ('onnx::loss::8', 'post_output')


2026-05-26 10:53:34.366031575 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer ConstantSharing modified: 0 with status: OK
2026-05-26 10:53:34.366101979 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer LayerNormFusionL1 modified: 0 with status: OK
2026-05-26 10:53:34.375994347 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer CommonSubexpressionElimination modified: 0 with status: OK
2026-05-26 10:53:34.376052576 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer GeluFusionL1 modified: 0 with status: OK
2026-05-26 10:53:34.376063558 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer SimplifiedLayerNormFusion modified: 0 with status: OK
2026-05-26 10:53:34.376071451 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer FastGeluFusion modified: 0 with status: OK
2026-05-26 10:53:34.376078859 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer

In [19]:
# Saving all the files to use them later for the training.
onnxblock.save_checkpoint(training_block.parameters(), artifacts_path / "checkpoint")

ir_version = 10
opset_import_version = 21

training_model.ir_version = ir_version
# print("Simplifying ONNX training model...")
# training_model, check = simplify(training_model)
# assert check, "Simplified ONNX training model could not be validated"artifacts_optim_2_comp_1
onnx.save(training_model, artifacts_path / "training_model.onnx")

optimizer_model.ir_version = ir_version
optimizer_model.opset_import[1].version = opset_import_version
onnx.save(optimizer_model, artifacts_path / "optimizer_model.onnx")

eval_model.ir_version = ir_version
# print("Simplifying ONNX eval model...")
# eval_model, check = simplify(eval_model)
# assert check, "Simplified ONNX eval model could not be validated"
onnx.save(eval_model, artifacts_path / "eval_model.onnx")

print(f"Artifacts saved in {artifacts_path} directory")

Artifacts saved in ../artifacts/fastvit_sa12/frozen_layer_9_classes_100/hailo8_optim_3_comp_0 directory


## Verify model

In [20]:
from pathlib import Path

from onnxruntime.training.api import CheckpointState, Module, Optimizer

In [21]:
# artifacts path
device = "cpu"
artifacts_path = Path(f"../artifacts/{model_name}/frozen_layer_{frozen_layer_num}_classes_{classes_num}/cpu")

print("Loading artifacts...")
# Create checkpoint state
state = CheckpointState.load_checkpoint(artifacts_path / "checkpoint")

# Create module
print(f"Creating model with {device} device...")
model = Module(
    artifacts_path / "training_model.onnx",
    state,
    artifacts_path / "eval_model.onnx",
    device=device,
)

# Create optimizer
optimizer = Optimizer(artifacts_path / "optimizer_model.onnx", model)
optimizer.set_learning_rate(0.001)

Loading artifacts...
Creating model with cpu device...


In [22]:
print(f"Model inputs: {model.input_names()}")
print(f"Model outputs: {model.output_names()}")

batch_size = 1
num_classes = 10  # Set this to your actual number of classes

# Prepare dummy input and label
forward_inputs = [
    np.ones((batch_size, 3, 224, 224), dtype=np.float32),  # input tensor
    np.zeros((batch_size,), dtype=np.int64),               # labels tensor
]

# If your model expects more inputs (check model.input_names()), add more dummy arrays as needed

model.train()
train_loss, logits = model(*forward_inputs)

Model inputs: ['input', 'labels']
Model outputs: ['onnx::loss::38', 'output']
